# 01 — Prepare CamemBERT Dataset

**Input:** raw question files (MedQuAD exports + manually written questions)

**Output:** `data/questions_dataset/dataset_clean.csv` — columns: `text`, `label`

**Labels:** `information_generale` · `sensible` · `avis_medical_personnalise` · `urgence` · `hors_perimetre`

---

## Pipeline steps
1. Load raw questions (MedQuAD + manual)
2. Clean text using `src/preprocessing/text_cleaner.py`
3. Assign labels (rule-based first, then manual review)
4. Check class balance
5. Stratified split: 70% train / 15% val / 15% test
6. Save CSVs to `data/questions_dataset/`

In [ ]:
import sys
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

sys.path.append('..')

DATASET_DIR = Path('../data/questions_dataset')
DATASET_DIR.mkdir(parents=True, exist_ok=True)

## Step 1 — Load raw questions

In [ ]:
# Load MedQuAD questions (select French-compatible ones)
# medquad_df = pd.read_csv(DATASET_DIR / 'medquad_raw.csv')

# Load manually written ICU questions
# manual_df = pd.read_csv(DATASET_DIR / 'icu_manual_questions.csv')

# Merge
# raw_df = pd.concat([medquad_df, manual_df], ignore_index=True)
# raw_df = raw_df[['text', 'label']]
# print(f'Total raw questions: {len(raw_df)}')
# raw_df.head()

## Step 2 — Clean text using TextCleaner

Using the same cleaner as runtime ensures training distribution matches inference distribution.

In [ ]:
from src.preprocessing.text_cleaner import TextCleaner

cleaner = TextCleaner()

# raw_df['text'] = raw_df['text'].apply(lambda t: cleaner.clean(t))
# Drop rows where cleaning returned empty/None
# raw_df = raw_df.dropna(subset=['text'])
# raw_df = raw_df[raw_df['text'].str.strip() != '']
# print(f'After cleaning: {len(raw_df)} questions')

## Step 3 — Assign labels

Rule-based annotation first (using `RuleFilter`), then manual correction.

Label mapping:
- `information_generale` — general ICU questions, no personalization
- `sensible` — personal phrasing but reformulable
- `avis_medical_personnalise` — diagnosis / prognosis / treatment request
- `urgence` — emergency situation
- `hors_perimetre` — outside chatbot scope

In [ ]:
from src.safety.rule_filter import RuleFilter

rule_filter = RuleFilter()

# def auto_label(text: str) -> str:
#     category, blocked = rule_filter.filter(text)
#     label_map = {
#         'urgency':       'urgence',
#         'diagnosis':     'avis_medical_personnalise',
#         'treatment':     'avis_medical_personnalise',
#         'prognosis':     'avis_medical_personnalise',
#         'personal_data': 'hors_perimetre',
#         'out_of_scope':  'hors_perimetre',
#         'allowed':       'information_generale',
#     }
#     return label_map.get(category, 'information_generale')

# raw_df['label_auto'] = raw_df['text'].apply(auto_label)
# raw_df.head(10)  # Review a sample before accepting labels

In [ ]:
# Manual review: export to CSV for review, re-import corrected version
# raw_df.to_csv(DATASET_DIR / 'to_review.csv', index=False)

# After manual correction:
# df = pd.read_csv(DATASET_DIR / 'to_review_corrected.csv')
# df = df[['text', 'label']]  # keep only final columns

## Step 4 — Check class balance

In [ ]:
# counts = df['label'].value_counts()
# print(counts)

# counts.plot(kind='bar', title='Class distribution', ylabel='Count', xlabel='Label')
# plt.tight_layout()
# plt.show()

# WARNING: Flag if any class has < 30 examples — consider oversampling or adding data

## Step 5 — Stratified train / val / test split (70 / 15 / 15)

In [ ]:
from sklearn.model_selection import train_test_split

# X = df['text']
# y = df['label']

# X_train, X_tmp, y_train, y_tmp = train_test_split(
#     X, y, test_size=0.30, stratify=y, random_state=42
# )
# X_val, X_test, y_val, y_test = train_test_split(
#     X_tmp, y_tmp, test_size=0.50, stratify=y_tmp, random_state=42
# )

# train_df = pd.DataFrame({'text': X_train, 'label': y_train})
# val_df   = pd.DataFrame({'text': X_val,   'label': y_val})
# test_df  = pd.DataFrame({'text': X_test,  'label': y_test})

# print(f'Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}')

## Step 6 — Save datasets

In [ ]:
# df.to_csv(DATASET_DIR / 'dataset_clean.csv', index=False)
# train_df.to_csv(DATASET_DIR / 'train.csv',   index=False)
# val_df.to_csv(DATASET_DIR / 'val.csv',       index=False)
# test_df.to_csv(DATASET_DIR / 'test.csv',     index=False)

# print('Saved: dataset_clean.csv, train.csv, val.csv, test.csv')